> **Release dependency:** the original `clinical_review/` helper package was absent from the supplied source tree. This notebook is retained for documentation but cannot run until those original helpers are restored. See `docs/REPRODUCIBILITY.md`. No completed reviewer data are distributed.

# BR08BS — Import completed same-reviewer adjudications

Upload `adjudications_completed.json` to the existing calibration package's `PUBLIC` directory. Run the cells below in order in the CPU review/evaluation kernel. No generator, GPU or new helper script is needed.

This notebook validates every decision before writing any adjudication, checks the frozen initial ratings and queue, preserves an exact copy of the upload, and calls the existing native `save_adjudication` function. Identical existing decisions are skipped; conflicting decisions stop the whole import. Interrupted imports can resume with the same file.

For `retain_initial`, `final_rating: null` means the native workflow copies the initial rating unchanged. For `revise_initial`, a complete valid revised rating is required. For `unresolved`, the final rating remains null. No clinical judgment is generated by this notebook. Keep existing Python helpers, frozen sources, original ratings and interpretations unchanged.

In [ ]:
from pathlib import Path
import sys
import json
import hashlib
import os
from collections import Counter
from datetime import datetime, timezone

EXP = Path('/path/to/repository/experiment02')
REVIEW_DIR = Path(
    '/path/to/repository/outputs/real_run_001/'
    'experiment02_v1/context_fit/v1/shaoxing/clinical_review/shaoxing/calibration_v1'
)
INPUT_PATH = REVIEW_DIR / 'PUBLIC/adjudications_completed.json'
if not (EXP / 'common.py').is_file():
    raise FileNotFoundError('Set EXP to the experiment02 folder containing common.py')
sys.path.insert(0, str(EXP))
from common import read, sha, signature, immutable, lock
from clinical_review.workflow import (
    _check, RATING_FIELDS, validate_rating, save_adjudication, progress
)
if not INPUT_PATH.is_file():
    raise FileNotFoundError('Upload completed decisions to: ' + str(INPUT_PATH))
print('Kernel:', sys.executable)
print('Review package:', REVIEW_DIR)
print('Upload:', INPUT_PATH)
print('Current progress:', progress(REVIEW_DIR))


## 1. Validate the complete upload and frozen evidence

The queue and image identities below were calculated from the downloaded PUBLIC package. Do not change the constants to bypass a mismatch. This notebook is bound to this calibration package. Validation checks every queued ID, initial-rating signature, reviewer, decision, rationale, evidence and final-rating rule. It also checks native frozen adjudication protocol hashes and rejects conflicting saved decisions before writing anything.

Validation confirms file identity and schema consistency; the clinician remains responsible for the clinical content. Only aggregate counts are printed; model identities are not displayed.

In [ ]:
EXPECTED_QUEUE_SIGNATURE = '69e5c68238db3b5a2921a9d591f7a3fb567487f1be456348ced4eeff4788a955'
EXPECTED_IMAGE_HASHES = {'images/case_00002.png': 'ee39eca22d36ae643fb20d14fc1f363098ee102d9f66ae6fc996f40aa32e06b9', 'images/case_00007.png': 'c284a0ae85bfcc4e7817eda2860abb8760f13cf300b89f66dbb110518810d562', 'images/case_00003.png': 'd9834e74e40b537a9829684724c12f3a6bcc5ef764a35bbc78d2e5499a727bc0', 'images/case_00010.png': 'd38b36ef60ef7c278d70db1c83f9e50cc547277cd6932692815b5f11e1d3c855', 'images/case_00008.png': '5b0daf107aa91e3f15b96cec4a57a65004cba0be228e70a16e603258f4892f7e', 'images/case_00001.png': '6db8426223a61dad249521cb821fcf82addfcaa601d281f4f830548cecc327f7', 'images/case_00005.png': '60b00d994aced0da2f638bc588679624c97d549c171c3bed183a3e47c51ebc66', 'images/case_00006.png': '9b38b2dca3f90c81db649a5383c84163ae1b78755820ee034abc5cbe4fe62d59', 'images/case_00009.png': '85a00b3a9485e8a6cd24f5fbe98865b99d52fa92f44cb9a5576afd571379e569', 'images/case_00004.png': '39e12bbd91b76265ec00781e26df293a63cbb9079b277510a8ea46aa45c349bc'}

def _unique_rows(rows, key):
    if not isinstance(rows, list) or not all(isinstance(r, dict) for r in rows):
        raise ValueError('Expected a list of objects: ' + key)
    ids = [r.get(key) for r in rows]
    if any(not isinstance(i, str) for i in ids) or len(set(ids)) != len(ids):
        raise ValueError('Missing or duplicate identifiers: ' + key)
    return {r[key]: r for r in rows}


def validate_completed_adjudications():
    protocol = _check(REVIEW_DIR)
    paths = {name: REVIEW_DIR / name for name in (
        'PRIVATE/initial_submitted.json', 'PRIVATE/references_submitted.json',
        'PRIVATE/adjudication_protocol.json', 'PUBLIC/adjudication_queue.json')}
    for name, path in paths.items():
        if not path.is_file():
            raise ValueError('Finish BR08A initial submission/queue preparation first: ' + name)
    ap = read(paths['PRIVATE/adjudication_protocol.json'])
    for field, name in (
        ('initial_sha256', 'PRIVATE/initial_submitted.json'),
        ('references_sha256', 'PRIVATE/references_submitted.json'),
        ('queue_sha256', 'PUBLIC/adjudication_queue.json')):
        if ap.get(field) != sha(paths[name]):
            raise ValueError('Frozen adjudication source changed: ' + name)
    if ap.get('reviewer_id') != protocol['reviewer_id']:
        raise ValueError('Frozen adjudication reviewer mismatch')
    queue = read(paths['PUBLIC/adjudication_queue.json'])
    queue_signature = signature(queue)
    if queue_signature != EXPECTED_QUEUE_SIGNATURE:
        raise ValueError('Queue differs from the downloaded clinician package')
    for relative, expected_sha in EXPECTED_IMAGE_HASHES.items():
        if sha(REVIEW_DIR / 'PUBLIC' / relative) != expected_sha:
            raise ValueError('Reviewed ECG image mismatch: ' + relative)
    queued = _unique_rows(queue, 'review_id')
    initial = _unique_rows(read(paths['PRIVATE/initial_submitted.json']), 'review_id')
    refs = _unique_rows(read(paths['PRIVATE/references_submitted.json']), 'case_id')
    reports = _unique_rows(read(REVIEW_DIR / 'PUBLIC/reports.json'), 'review_id')
    if set(initial) != set(reports):
        raise ValueError('Frozen initial rating identifiers do not match released reports')
    for rid, item in queued.items():
        if rid not in initial or item['initial_rating'] != initial[rid]:
            raise ValueError('Queue initial rating differs from frozen submission: ' + rid)
        if item['independent_interpretation'] != refs.get(item['case_id']):
            raise ValueError('Queue interpretation differs from frozen reference: ' + rid)
        if any(item.get(k) != reports[rid].get(k)
               for k in ('case_id', 'image', 'generated_report')):
            raise ValueError('Queued report differs from released report: ' + rid)
    payload = INPUT_PATH.read_bytes()
    source_sha = hashlib.sha256(payload).hexdigest()
    def unique_keys(pairs):
        result = {}
        for key, value in pairs:
            if key in result:
                raise ValueError('Duplicate JSON key: ' + key)
            result[key] = value
        return result
    def invalid_constant(value):
        raise ValueError('Nonstandard JSON number: ' + value)
    data = json.loads(payload, object_pairs_hook=unique_keys, parse_constant=invalid_constant)
    if not isinstance(data, dict) or data.get('schema_version') != 'blinded_report_adjudications_v1':
        raise ValueError('Unexpected adjudication upload schema')
    if data.get('stage') != protocol.get('stage'):
        raise ValueError('Upload stage differs from native package')
    if (data.get('source_queue_file') != 'adjudication_queue.json'
            or data.get('source_queue_signature') != queue_signature):
        raise ValueError('Upload does not identify this frozen queue')
    rows = data.get('adjudications')
    uploaded = _unique_rows(rows, 'review_id')
    if set(uploaded) != set(queued):
        raise ValueError('Provide each queued report exactly once; no missing or extra IDs')
    if (type(data.get('expected_adjudications')) is not int
            or data['expected_adjudications'] != len(queue)):
        raise ValueError('Expected adjudication count mismatch')
    snapshot = REVIEW_DIR / 'PRIVATE/imported_adjudications.json'
    if snapshot.exists() and sha(snapshot) != source_sha:
        raise ValueError('A different upload has already been imported; preserve the original')
    fields = {'review_id', 'case_id', 'image', 'reviewer_id', 'initial_rating_sha256',
              'decision', 'reason', 'evidence_reviewed', 'final_rating'}
    records, existing = [], set()
    for row in rows:
        rid = row['review_id']
        item = queued[rid]
        if set(row) != fields:
            raise ValueError('Unexpected/missing adjudication fields: ' + rid)
        if any(row[k] != item[k] for k in ('case_id', 'image')):
            raise ValueError('Case/image linkage mismatch: ' + rid)
        if row['reviewer_id'] != protocol['reviewer_id']:
            raise ValueError('Different reviewer: ' + rid)
        initial_sha = signature(item['initial_rating'])
        if row['initial_rating_sha256'] != initial_sha:
            raise ValueError('Initial rating signature mismatch: ' + rid)
        decision = row['decision']
        if decision not in ('retain_initial', 'revise_initial', 'unresolved'):
            raise ValueError('Invalid/missing adjudication decision: ' + rid)
        for field in ('reason', 'evidence_reviewed'):
            if not isinstance(row[field], str) or not row[field].strip():
                raise ValueError('Missing ' + field + ': ' + rid)
        original = {k: item['initial_rating'][k] for k in RATING_FIELDS}
        validate_rating(original, protocol['reviewer_id'])
        final = row['final_rating']
        if decision == 'retain_initial':
            if final is not None and final != original:
                raise ValueError('Retain cannot change the initial rating: ' + rid)
            final = original
        elif decision == 'revise_initial':
            if not isinstance(final, dict):
                raise ValueError('Revision requires a complete final_rating object: ' + rid)
            try:
                validate_rating(final, protocol['reviewer_id'])
            except (ValueError, TypeError) as exc:
                raise ValueError(rid + ': ' + str(exc)) from exc
            if final == original:
                raise ValueError('Revision must change a rating; otherwise retain: ' + rid)
        elif final is not None:
            raise ValueError('Unresolved must leave final_rating null: ' + rid)
        record = dict(review_id=rid, reviewer_id=row['reviewer_id'], decision=decision,
                      reason=row['reason'], evidence_reviewed=row['evidence_reviewed'],
                      final_rating=final, initial_rating_sha256=initial_sha)
        destination = REVIEW_DIR / 'PUBLIC/adjudications' / (rid + '.json')
        if destination.exists():
            saved = read(destination)
            if (set(saved) != set(record) | {'submitted_at_utc'}
                    or {k: saved.get(k) for k in record} != record):
                raise ValueError('Conflicting saved adjudication; no overwrite: ' + rid)
            existing.add(rid)
        records.append(record)
    return protocol, records, existing, payload, source_sha

protocol, records, existing, payload, source_sha = validate_completed_adjudications()
print('VALID:', len(records), 'completed adjudications')
print('Decisions:', dict(Counter(r['decision'] for r in records)))
print('Identical previously saved adjudications:', len(existing))
print('No adjudications have been written by validation.')


## 2. Import the validated clinician decisions

Running this cell submits the decisions through the native immutable workflow. It revalidates under a single-writer lock before any write and saves an exact private upload copy and audit log. It does not revise initial assessments, reinterpret the ECGs or resolve uncertainty automatically.

Do not edit the upload during import or run other BR08 adjudication-writing cells in parallel. An interruption is recoverable by rerunning this notebook with the identical file. Once saved, decisions cannot be replaced silently; corrections require an explicit new review version.

In [ ]:
def import_completed_adjudications():
    with lock(REVIEW_DIR / 'adjudication_import.lock'):
        protocol, records, existing, payload, source_sha = validate_completed_adjudications()
        snapshot = REVIEW_DIR / 'PRIVATE/imported_adjudications.json'
        if not snapshot.exists():
            temporary = snapshot.with_name(snapshot.name + '.tmp')
            with temporary.open('wb') as stream:
                stream.write(payload)
                stream.flush()
                os.fsync(stream.fileno())
            temporary.replace(snapshot)
        imported = 0
        for record in records:
            if record['review_id'] in existing:
                continue
            save_adjudication(REVIEW_DIR, record['review_id'],
                reviewer_id=record['reviewer_id'], decision=record['decision'],
                reason=record['reason'], evidence_reviewed=record['evidence_reviewed'],
                final_rating=record['final_rating'])
            imported += 1
        log_path = REVIEW_DIR / 'PRIVATE/adjudication_notebook_import.json'
        timestamp = (read(log_path)['imported_at_utc'] if log_path.exists()
                     else datetime.now(timezone.utc).isoformat())
        immutable(log_path, dict(
            workflow='BR08BS_import_completed_adjudications_v1', imported_at_utc=timestamp,
            source_sha256=source_sha, review_ids=[r['review_id'] for r in records],
            reviewer_id=protocol['reviewer_id'], queue_signature=EXPECTED_QUEUE_SIGNATURE,
            package_protocol_sha256=sha(REVIEW_DIR / 'PRIVATE/protocol.json'),
            adjudication_protocol_sha256=sha(REVIEW_DIR / 'PRIVATE/adjudication_protocol.json'),
            image_hashes=EXPECTED_IMAGE_HASHES,
            decision_counts=dict(Counter(r['decision'] for r in records)),
            initial_ratings_modified=False))
        # Recheck source identities and every saved decision, not merely file counts.
        _, checked, saved, _, _ = validate_completed_adjudications()
        if len(saved) != len(checked):
            raise ValueError('Some required adjudications are not saved')
    return dict(new_adjudications_imported=imported,
                identical_existing_adjudications=len(existing),
                decisions=dict(Counter(r['decision'] for r in records)),
                progress=progress(REVIEW_DIR))

print(import_completed_adjudications())


## 3. Check readiness for BR09

This verifies every saved decision against the uploaded decisions and frozen queue. Unresolved entries are registered explicitly rather than assigned fabricated final ratings. Readiness means the records are complete for analysis; it does not establish clinical validity or independent reviewer agreement.

In [ ]:
if not (REVIEW_DIR / 'PRIVATE/adjudication_notebook_import.json').is_file():
    raise ValueError('Run the import cell first')
_, records, existing, _, _ = validate_completed_adjudications()
if len(existing) != len(records):
    raise ValueError('Required adjudications remain unsaved; do not run BR09 yet')
print('READY FOR BR09:', progress(REVIEW_DIR))
print('Decision counts:', dict(Counter(r['decision'] for r in records)))
print('Saved decisions:', REVIEW_DIR / 'PUBLIC/adjudications')
print('Continue to BR09 using the SAME calibration_v1 review directory.')
print('Preserve single-reviewer, unresolved-case and calibration-sample limitations.')
